<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/04_dynamic_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R4 動的計画法:方策評価・方策反復・価値反復(notebook)

本文:[`04-dynamic-programming.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/04-dynamic-programming.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

# R2〜R5 で使うグリッド(本文 R2 §3)
N_ROW = N_COL = 4
S, A = N_ROW * N_COL, 4
START, GOAL, MUD = 0, 15, {5, 10}
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # 0=上、1=右、2=下、3=左
ARROWS = ["↑", "→", "↓", "←"]

def to_pos(s):
    return divmod(s, N_COL)

def next_state(s, a):
    # 行動 a の向きに1マス進む。外に出るなら、その場にとどまる(R2 §2)
    r, c = to_pos(s)
    dr, dc = MOVES[a]
    r2, c2 = r + dr, c + dc
    return r2 * N_COL + c2 if (0 <= r2 < N_ROW and 0 <= c2 < N_COL) else s

def build_grid_mdp(slip=0.2, mud_reward=-10.0):
    P = np.zeros((S, A, S))
    R = np.zeros((S, A))
    for s in range(S):
        for a in range(A):
            if s == GOAL:                       # ゴールは吸収状態:とどまり、報酬 0
                P[s, a, s] = 1.0
                continue
            for a2 in range(A):                 # 実際に進む方向 a2 の確率
                p = (1 - slip) * (a2 == a) + slip / A
                P[s, a, next_state(s, a2)] += p
            R[s, a] = sum(P[s, a, s2] * (mud_reward if s2 in MUD else -1.0) for s2 in range(S))
    return P, R

def policy_value(P, R, policy, gamma):
    # 決定的な方策の価値を (I - gamma P_pi) v = r_pi で求める(R3)
    idx = np.arange(P.shape[0])
    return np.linalg.solve(np.eye(P.shape[0]) - gamma * P[idx, policy], R[idx, policy])

def show_policy(policy, title=""):
    # 方策を矢印で表示する(G = ゴール、ぬかるみのマスには * をつける)
    cells = []
    for s in range(S):
        mark = "G " if s == GOAL else ARROWS[policy[s]] + ("*" if s in MUD else " ")
        cells.append(mark)
    print(title)
    for r in range(N_ROW):
        print("  " + " ".join(cells[r * N_COL:(r + 1) * N_COL]))

def show_values(V, title=""):
    plt.figure(figsize=(3.6, 3.2))
    plt.imshow(V.reshape(N_ROW, N_COL), cmap="viridis")
    for s in range(S):
        r, c = to_pos(s)
        plt.text(c, r, f"{V[s]:.1f}", ha="center", va="center", color="w", fontsize=9)
    plt.colorbar(); plt.title(title); plt.xticks([]); plt.yticks([]); plt.show()

gamma = 0.95
P, R = build_grid_mdp(slip=0.2)
right_then_down = np.array([1 if to_pos(s)[1] < N_COL - 1 else 2 for s in range(S)])
print("R2・R3 のグリッド(滑る床 0.2、ぬかるみ -10、γ = 0.95)の P と R を用意した:", P.shape, R.shape)

## 1. 反復方策評価

**予想**:R3 で、方策「右→下」の価値を `solve` で求めた。同じ値を、式 (1) のくり返しで求めると、変化が $10^{-10}$ より小さくなるまでに、何回くらいかかるか。ほとんどゴールに着かない方策「いつも右」ではどうか。

- 予想:(ここに書く)

**実装**:`policy_evaluation` を書く(本文 §1)。

In [ ]:
def policy_evaluation(P, R, policy, gamma, tol=1e-10):
    raise NotImplementedError("TODO: V = 0 から始め、V_new = (R + gamma * P @ V)[idx, policy] をくり返す。"
                              "変化 max|V_new - V| を deltas に記録し、tol より小さくなったら V と deltas を返す")

always_right = np.full(S, 1)
V_pe, deltas = policy_evaluation(P, R, right_then_down, gamma)
V_ar, deltas_ar = policy_evaluation(P, R, always_right, gamma)
check("くり返して求めた価値 = solve で求めた価値(右→下)", V_pe, policy_value(P, R, right_then_down, gamma), tol=1e-7)
check("くり返して求めた価値 = solve で求めた価値(いつも右)", V_ar, policy_value(P, R, always_right, gamma), tol=1e-7)
print("くり返しの回数:右→下", len(deltas), "回、いつも右", len(deltas_ar), "回")
print("上限の目安 log(1e-10)/log(γ) =", round(np.log(1e-10) / np.log(gamma)), "回")

In [ ]:
plt.figure(figsize=(5, 3.2))
plt.semilogy(deltas, label="right then down")
plt.semilogy(deltas_ar, label="always right")
k = np.arange(len(deltas_ar))
plt.semilogy(deltas_ar[0] * gamma ** k, "k--", label="gamma^k (upper bound)")
plt.xlabel("iteration k"); plt.ylabel("change (log scale)"); plt.legend(); plt.show()

破線の $\gamma^k$ は、誤差の縮み方の**上限**である(本文 §1)。すぐにゴールに着く「右→下」では、まだゴールに着いていない確率の分だけ速く縮むので、破線よりずっと早く小さくなる。ほとんどゴールに着かない「いつも右」は、破線に近い遅さでしか小さくならない。

## 2. 方策改善

**実装**:価値 `V` に対する貪欲な方策を返す関数 `greedy_policy` を書く(本文の式 (2))。「いつも右」の方策を1回改善して、価値が下がらないことを確かめる(方策改善定理)。

In [ ]:
def greedy_policy(P, R, V, gamma):
    raise NotImplementedError("TODO: Q = R + gamma * P @ V の、各状態で最大の行動(argmax)")

pi0 = np.full(S, 1)                                   # 最初の方策:いつも右
V0 = policy_value(P, R, pi0, gamma)
pi1 = greedy_policy(P, R, V0, gamma)
V1 = policy_value(P, R, pi1, gamma)
check("改善した方策は、すべてのマスで同じかそれ以上の価値(方策改善定理)", np.all(V1 >= V0 - 1e-9), True)
show_policy(pi0, "改善の前(いつも右)")
show_policy(pi1, "1回改善したあと")
print(f"スタートの価値:{V0[START]:.2f} → {V1[START]:.2f}")

## 3. 方策反復

**予想**:評価と改善を、方策が変わらなくなるまでくり返す。外側のくり返しは何回くらいで終わるか。

- 予想:(ここに書く)

**実装**:`policy_iteration` を書く(本文 §3)。評価には、R3 の `policy_value`(`solve` で厳密に解く)を使う。

In [ ]:
def policy_iteration(P, R, gamma):
    raise NotImplementedError("TODO: いつも右の方策から始め、評価(policy_value)と改善(greedy_policy)をくり返す。"
                              "方策が変わらなくなったら、方策・価値・外側のくり返しの回数を返す")

pi_pi, V_pi, n_outer = policy_iteration(P, R, gamma)
rhs = (R + gamma * P @ V_pi).max(axis=1)
check("方策反復の価値は、ベルマン最適方程式を満たす", V_pi, rhs, tol=1e-8)
print("外側のくり返しの回数:", n_outer)

## 4. 価値反復

**実装**:`value_iteration` を書く(本文 §4)。方策反復と同じ最適な価値になることと、求めた方策の価値が $V^{\ast}$ に一致することを確かめる。

In [ ]:
def value_iteration(P, R, gamma, tol=1e-10):
    raise NotImplementedError("TODO: V = 0 から始め、Q = R + gamma * P @ V、V_new = Q.max(axis=1) をくり返す。"
                              "変化が tol より小さくなったら、V_new・Q.argmax(axis=1)・くり返しの回数を返す")

V_vi, pi_vi, n_sweeps = value_iteration(P, R, gamma)
check("価値反復の V* = 方策反復の V*", V_vi, V_pi, tol=1e-7)
check("価値反復で求めた方策の価値 = V*", policy_value(P, R, pi_vi, gamma), V_vi, tol=1e-7)
print("価値反復のくり返しの回数:", n_sweeps, "  方策反復の外側のくり返しの回数:", n_outer)

## 5. 最適な方策を読む

**予想**:ぬかるみ(マス 5 と 10)の隣のマスで、最適な方策はどちらに進むか。滑らない床ではどうか(本文の確認課題5)。

- 予想:(ここに書く)

In [ ]:
show_policy(pi_vi, "最適な方策(滑る床 0.2)。* はぬかるみ")
show_values(V_vi, "V* (slip = 0.2)")

P0, R0 = build_grid_mdp(slip=0.0)
V0_star, pi0_star, _ = value_iteration(P0, R0, gamma)
show_policy(pi0_star, "最適な方策(滑らない床)")
Q_slip = R + gamma * P @ V_vi
Q_flat = R0 + gamma * P0 @ V0_star
for s in range(S):
    if s == GOAL:
        continue
    best_flat = {a for a in range(A) if Q_flat[s, a] > V0_star[s] - 1e-9}   # 最適な行動は複数あることがある
    best_slip = {a for a in range(A) if Q_slip[s, a] > V_vi[s] - 1e-9}
    if best_flat - best_slip:
        print(f"マス {s}:滑らない床で最適な行動 {[ARROWS[a] for a in sorted(best_flat)]}、"
              f"滑る床で最適な行動 {[ARROWS[a] for a in sorted(best_slip)]}")

滑らない床では、ぬかるみに入らない限り、どの最短経路も同じ価値になる。そのため、マス 2 とマス 8 では「右」と「下」のどちらも最適である。滑る床では、ぬかるみの隣のマス(マス 6 とマス 9)に入ると、そこから滑ってぬかるみに入る危険がある。そのため、マス 2 では「下」(マス 6 へ)が、マス 8 では「右」(マス 9 へ)が、最適ではなくなる。

## 6. 割引率と収束の速さ

**予想**:$\gamma$ を 0.5、0.9、0.95、0.99 と大きくすると、価値反復のくり返しの回数はどう変わるか。方策「いつも右」の反復方策評価の回数はどうか(本文の確認課題3)。

- 予想:(ここに書く)

In [ ]:
for g in [0.5, 0.9, 0.95, 0.99]:
    V_g, pi_g, n_g = value_iteration(P, R, g)
    n_ar = len(policy_evaluation(P, R, always_right, g)[1])
    print(f"γ = {g:4}:価値反復 {n_g:4d} 回、いつも右の評価 {n_ar:5d} 回(上限の目安 {np.log(1e-10) / np.log(g):5.0f} 回)"
          f"   V*(スタート) = {V_g[START]:6.2f}")

価値反復では、最適な方策がすぐにゴールに着くので、$\gamma$ を大きくしても回数はあまり増えない。一方、ゴールにほとんど着かない「いつも右」の評価では、$\gamma$ が 1 に近いほど回数が大きく増える。

## 7. 条件を変える

1. ぬかるみの報酬を −10 から −2 にしても(`build_grid_mdp(0.2, mud_reward=-2.0)`)、最適な方策がぬかるみを通らないのはなぜか。報酬を 0(ふつうのマスの −1 より得)にすると、どうなるか
2. `slip` を 0.4 にすると、最適な方策はどう変わるか
3. 本文の確認課題1・2:R3 §6 の小さな MDP で、`policy_iteration` と `value_iteration` を動かす(R3 の `P_small`・`R_small` をここに書き写す)

In [ ]:
# 1〜3 をここで試す

## 8. 研究室のライブラリ `cpai_lab` との関係

`cpai_lab` の `value_iteration` と `policy_evaluation` は、この単元と同じ計算をする。研究では、学習した結果を確かめるための「正解」の計算に使う。自分の実装と同じ結果になることを確かめる。

In [ ]:
# 研究室のライブラリ cpai_lab を入れる(Colab では、ランタイムをつなぐたびに必要)
%pip install -q git+https://github.com/oit-cpai/lab-core.git

In [ ]:
from cpai_lab import value_iteration as lab_value_iteration, policy_evaluation as lab_policy_evaluation

V_lab, Q_lab, pi_lab = lab_value_iteration(P, R, gamma=gamma, tol=1e-10)
check("cpai_lab の value_iteration の V* = 自分の実装の V*", V_lab, V_vi, tol=1e-7)
check("cpai_lab の policy_evaluation = 自分の実装", lab_policy_evaluation(P, R, right_then_down, gamma=gamma, tol=1e-10),
      policy_value(P, R, right_then_down, gamma), tol=1e-7)

`GridMazeEnv` の `transition_model()` は、迷路の環境から `P` と `R` をそのまま作る。R2 の最後の節と同じ 4×4 の広場で、ぬかるみのない場合の最適な価値が、自分で作った `P`・`R`(ぬかるみの報酬を −1 にしたもの)と一致することを確かめる。

In [ ]:
from cpai_lab import GridMazeEnv
from cpai_lab.envs.grid_maze import grid_from_text

grid, start, goal, _ = grid_from_text("""
######
#S...#
#....#
#....#
#...G#
######
""")
lab_env = GridMazeEnv(grid=grid, start=start, goal=goal, slip_prob=0.2, step_reward=-1.0, goal_reward=-1.0)
P_lab, R_lab = lab_env.transition_model()
print("cpai_lab の P の形(壁も含めた 6×6 = 36 マス):", P_lab.shape)
V_env, _, pi_env = lab_value_iteration(P_lab, R_lab, gamma=gamma, tol=1e-10)

P_nomud, R_nomud = build_grid_mdp(0.2, mud_reward=-1.0)
V_nomud, _, _ = value_iteration(P_nomud, R_nomud, gamma)
check("スタートの V*:cpai_lab の環境 = 自分の P・R", V_env[lab_env.pos_to_obs(start)], V_nomud[START], tol=1e-6)
print("cpai_lab の行動の番号は 0=上、1=下、2=左、3=右。スタートでの最適な行動:", pi_env[lab_env.pos_to_obs(start)])

## 9. 説明(自分の言葉で書く)

1. 反復方策評価で、くり返すと価値が正しい値に近づくのはなぜか。割引率が 1 に近いと、何が困るか
   - (ここに書く)
2. 方策反復と価値反復の違いは何か。それぞれ、何をくり返しているか
   - (ここに書く)
3. 滑る床で、最適な方策がぬかるみから離れるのはなぜか
   - (ここに書く)